In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

from mcfs.load_runs import load_saved_fake_spectra
import numpy as np
import mcfs

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = mcfs.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
axis = 1

data = load_saved_fake_spectra(
    hdf5_path="/home/STORAGE/SKEWERS/TNG50-2/snapdir_033/axis_"+str(axis)+"/grid_TNG50-2_snap033_nspec512_axis"+str(axis)+"_nbins1024_lya_si.hdf5",
    base="/home/STORAGE/TNG50-2",
    num=33,
    axis=-1,
    use_res=None,
)

In [ ]:
print("\n=== Loaded paths ===")
for k, v in data["paths"].items():
    print(f"{k:10s}: {v}")

print("\n=== Loaded lines ===")
print(data["lines"])

print("\n=== fake_spectra metadata ===")
for k, v in data["meta"].items():
    print(f"{k:10s}: {v}")

print("\n=== Timing info ===")
for k, v in data["timing"].items():
    print(f"{k:18s}: {v}")

print("\n=== Array keys ===")
print("tau :", list(data["tau"].keys()))
print("n   :", list(data["n"].keys()))
print("T   :", list(data["T"].keys()))
print("vlos:", list(data["vlos"].keys()))

In [ ]:
sp = data["sp"]
tau = data["tau"]
n = data["n"]
T = data["T"]
vlos = data["vlos"]
lines = data["lines"]

In [ ]:
L_box = 51.7  # Mpc

H0_km_s_mpc = 70.0
c_m_s = 2.99792458e8
H0_over_c = (H0_km_s_mpc * 1e6) / c_m_s * 1e-3 # 1/Mpc
scale_factor_ratio = (1 - L_box * H0_over_c / 2 )**-2 # assuming DM domination

In [ ]:
line_info = {
    "H1_1215":   {"label": r"HI Ly$\alpha$",   "lambda0": 1215.67   , "tau_factor": 1e0},
    "Si3_1206":  {"label": r"SiIII 1206",      "lambda0": 1206.50   , "tau_factor": 1e5},
    "Si2_1190":  {"label": r"SiII 1190",       "lambda0": 1190.4158 , "tau_factor": 1e5},
    "Si2_1193":  {"label": r"SiII 1193",       "lambda0": 1193.2897 , "tau_factor": 1e5},
}

lambda0_lya = 1215.67
c_kms = 299792.458

# Common native grid from fake_spectra
dv = sp.dvbin
v_kms = np.arange(sp.nbins) * dv
keys_lines = [f"{e}{i}_{lam}" for (e, i, lam) in lines]

In [ ]:
# Build full joined x-domain covering all shifted lines
xmins = []
xmaxs = []
for k in keys_lines:
    if k not in tau:
        continue
    lambda0 = line_info[k]["lambda0"]
    delta_v = c_kms * np.log(lambda0 / lambda0_lya)
    data["delta_v_"+k] = delta_v
    x_line = v_kms + delta_v
    data["x_"+k] = x_line
    xmins.append(x_line.min())
    xmaxs.append(x_line.max())

# Use same spacing as original fake_spectra grid
x_total = np.arange(min(xmins), max(xmaxs) + dv, dv)

# infer number of skewers from first available tau field
first_key = next(k for k in keys_lines if k in tau)
n_skewers = tau[first_key].shape[0]

flux_lines_interp = {}
tau_total = np.zeros((n_skewers, x_total.size), dtype=float)
# Interpolate each shifted line onto x_total and sum
for k in keys_lines:
    for i in range(tau[k].shape[0]):
        tau_line = tau[k][i] * line_info[k]["tau_factor"]
        tau_interp = np.interp(x_total, data["x_"+k], tau_line, left=0.0, right=0.0)
        tau_total[i] += tau_interp

    tau_line = tau[k] * line_info[k]["tau_factor"] # shape: (N_skewers, nbins_native)
    flux_interp = np.zeros((tau_line.shape[0], x_total.size), dtype=float)
    for i in range(tau_line.shape[0]):
        flux_native = np.exp(-tau_line[i])
        flux_interp[i] = np.interp(x_total, data["x_" + k], flux_native, left=1.0, right=1.0)
    flux_lines_interp[k] = flux_interp

data["tau_total"] = tau_total
data["x_total"] = x_total
data["flux_lines_interp"] = flux_lines_interp

In [ ]:
linestyles = ["--", ":", "-.", (0, (3, 1, 1, 1)), (0, (5, 1))]
cmap = plt.get_cmap("tab20")

In [ ]:
# --- choose which skewers to plot ---
mode = "random"     # "random", "first", or "given"
Nplot = 2
seed = 137
axis_select = None  # 1, 2, 3, or None
ii_list = [0, 1, 2, 3]   # only used if mode == "given"

# --- allowed indices ---
all_idx = np.arange(sp.NumLos)

if axis_select is not None:
    if not hasattr(sp, "axis"):
        raise AttributeError("sp has no attribute 'axis' (needed for axis_select).")
    allowed = np.where(sp.axis == axis_select)[0]
    if allowed.size == 0:
        raise ValueError(f"No skewers found with axis_select={axis_select}.")
else:
    allowed = all_idx

# --- choose ii_list ---
if mode == "random":
    rng = np.random.default_rng(seed)
    ii_list = rng.choice(allowed, size=min(Nplot, allowed.size), replace=False)
elif mode == "first":
    ii_list = allowed[: min(Nplot, allowed.size)]
elif mode == "given":
    ii_list = np.array(ii_list, dtype=int)
    bad = np.setdiff1d(ii_list, allowed)
    if bad.size > 0:
        raise ValueError(f"Some indices are not allowed: {bad[:10]}")
else:
    raise ValueError("mode must be one of: 'random', 'first', 'given'.")

print(f"Plotting {len(ii_list)} skewers | mode={mode} | axis_select={axis_select}")
print("Indices:", ii_list[:30], "..." if len(ii_list) > 30 else "")

# --- print mean flux across ALL skewers/pixels ---
print(f"\nGlobal means over all skewers and pixels (NumLos={sp.NumLos}, nbins={sp.nbins}):")
for k in keys_lines:
    if k in tau:
        F = np.exp(-tau[k] * line_info[k]["tau_factor"])
        print(f"  {k:10s}  <F> = {float(F.mean()):.6f}")

print(f"  {'Total':10s}  <F> = {float(np.exp(-data['tau_total']).mean()):.6f}")

In [ ]:
xlabel = r"$c\, \ln\left( \frac{\lambda}{\lambda_\dagger}\right)\; \left[\mathrm{km\,/\,s}\right]$"

# --- plotting ---
cmap = plt.get_cmap("tab10")
colors = {k: cmap(i % cmap.N) for i, k in enumerate(keys_lines)}

N_cols = 2
for ii in ii_list:
    fig, axes = plt.subplots(N_cols, 1, figsize=(14, 3*N_cols), sharex=True)

    for k in keys_lines:
        axes[0].plot(v_kms, tau[k][ii]*line_info[k]["tau_factor"], color=colors[k], lw=1.5)
        axes[1].plot(v_kms, np.exp(-tau[k][ii]*line_info[k]["tau_factor"]), color=colors[k], lw=1.5)
        # axes[2].plot(v_kms, n[k][ii], color=colors[k], lw=1.5)
        # axes[3].plot(v_kms, T[k][ii], color=colors[k], lw=1.5)
        # axes[4].plot(v_kms, vlos[k][ii], color=colors[k], lw=1.5)

    axes[0].set_title(f"skewer" f" {ii}" f" | z={sp.red:.3f}")

    axes[0].set_ylabel(r"$\tau$")
    axes[0].set_yscale("log")

    axes[1].set_ylabel(r"$e^{-\tau}$")
    axes[1].set_xlabel(xlabel)

    # axes[2].set_ylabel(r"$n$ [cm$^{-3}$]")
    # axes[2].set_yscale("log")

    # axes[3].set_ylabel(r"$T$ [K]")
    # axes[3].set_yscale("log")

    # axes[4].set_ylabel(r"$v_{\parallel}$ [km/s]")

    # Legend: one entry per line
    handles = [Line2D([0], [0], color=colors[k], lw=3, label=k) for k in keys_lines]
    axes[0].legend(handles=handles, loc="upper right", frameon=True, fontsize=12, title="Line key")

    plt.tight_layout()
    plt.show()

In [ ]:
xlabel = r"$c\, \ln\left( \frac{\lambda}{\lambda_{\mathrm{Ly}\alpha}}\right)\; \left[\mathrm{km\,/\,s}\right]$"

# Plot: one row per skewer, flux only
N_plot_sample = len(ii_list)
fig, axes = plt.subplots(N_plot_sample, 1, figsize=(14, 2.0 * N_plot_sample), sharex=True, sharey=True)

if N_plot_sample == 1:
    axes = [axes]

# one color per skewer
colors_skewer = {idx: cmap(i % cmap.N) for i, idx in enumerate(ii_list)}
list_handles = [Line2D([0], [0], color="k", lw=1.5, linestyle="-", label="Total")]
for ii, idx in enumerate(ii_list):
    ax = axes[ii]
    skewer_color = colors_skewer[idx]

    # total flux on common joined axis
    ax.plot(data["x_total"], np.exp(-data["tau_total"][idx]), color=skewer_color, lw=1.5, linestyle="-")

    # individual line fluxes on shifted axes
    for i_line, k in enumerate(keys_lines):
        
        tau_factor = line_info[k]["tau_factor"]

        flux_line = np.exp(-tau[k][idx] * tau_factor)
        ls = linestyles[i_line % len(linestyles)]
        ax.plot(data["x_"+k], flux_line, color=skewer_color, lw=2.5, linestyle=ls)

        ax.axvline(data["delta_v_"+k], color="grey", linestyle=ls, lw=1.5)
        # ax.axvline(c_kms * np.log(scale_factor_ratio * line_info[k]["lambda0"] / lambda0_lya), color="grey", linestyle=ls, lw=1.5)

        if ii == 0:
            label = line_info[k].get("label", k)
            list_handles.append(Line2D([0], [0], color="k", lw=1.5, linestyle=ls, label=label))

    ax.set_ylim([-0.05, 1.05])
    ax.set_ylabel(r"$e^{-\tau}$", fontsize=16)

    ax.text(0.015, 0.88, f"Skewer {idx}", transform=ax.transAxes, fontsize=12, verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor=skewer_color)
    )

axes[-1].set_xlabel(xlabel, fontsize=18)
axes[0].set_title(f"{len(ii_list)} skewers | z={sp.red:.3f}", fontsize=18)

fig.legend(
    handles=list_handles, loc="upper center", bbox_to_anchor=(0.5, 0.995), ncols=len(keys_lines) + 1,
    frameon=True, fontsize=13, handlelength=2.6, handletextpad=0.6,
)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
import mcfs.xi1D as xi1D
import jax.numpy as jnp

In [ ]:
# ------------------------------------------------------------
# 2PCF settings
# ------------------------------------------------------------
n_bins = 256
r_min = 0.0
r_max = float(data["x_total"].max() - data["x_total"].min())
include_self = False
center = True
standardize = False
chunk_size = 1 << 15

In [ ]:
# 1) Precompute geometry ONCE on the common joined axis
pb = xi1D.precompute_pair_binning_1d(
    data["x_total"], n_bins=n_bins, r_min=r_min, r_max=r_max, include_self=include_self, chunk_size=chunk_size,
)
r_centers = pb.r_centers

xi_auto = {}
xi_cross = {}

# 2) Auto for total
xi_auto["total"] = xi1D.two_point_corr_1d_from_precomp(
    pb, jnp.exp(-data["tau_total"]), center=center, standardize=standardize,
)

# 3) Auto for each line
for k in keys_lines:
    xi_auto[k] = xi1D.two_point_corr_1d_from_precomp(
        pb, jnp.asarray(flux_lines_interp[k]), center=center, standardize=standardize,
    )

# 4) Cross only for unique pairs i<j
for i, ki in enumerate(keys_lines):
    xi_cross[ki] = {}
    for j in range(i + 1, len(keys_lines)):
        kj = keys_lines[j]
        xi_cross[ki][kj] = xi1D.two_point_corr_1d_cross_from_precomp(
            pb, jnp.asarray(flux_lines_interp[ki]), jnp.asarray(flux_lines_interp[kj]),
            center=center, standardize=standardize, symmetrize=True,
        )

In [ ]:
x_label = r"$\Delta\!\left[c\,\ln\!\left(\lambda/\lambda_{\mathrm{Ly}\alpha}\right)\right]\;[\mathrm{km\,s^{-1}}]$"

# ------------------------------------------------------------
# Collect unique cross pairs
# ------------------------------------------------------------
pairs = []
for name_i in xi_cross:
    for name_j in xi_cross[name_i]:
        pairs.append((name_i, name_j))

fig, ax = plt.subplots(figsize=(15, 7.0))

# ------------------------------------------------------------
# Color maps
# ------------------------------------------------------------
auto_names = list(keys_lines)

auto_cmap = plt.get_cmap("tab10")
auto_colors = {name: auto_cmap(i % 10) for i, name in enumerate(auto_names)}

total_color = "k"

n_pairs = max(len(pairs), 1)
cross_cmap = plt.get_cmap("tab20")
cross_colors = {
    pair: cross_cmap(t)
    for pair, t in zip(pairs, np.linspace(0.0, 1.0, n_pairs, endpoint=False))
}

# ------------------------------------------------------------
# 1) TOTAL: solid black, mean ± std
# ------------------------------------------------------------
x_tot = np.asarray(xi_auto["total"])   # shape: (N_skewers, n_bins)
mean_tot = np.nanmean(x_tot, axis=0)
std_tot  = np.nanstd(x_tot, axis=0)

ax.plot(r_centers, mean_tot, lw=2.8, c=total_color, ls="-", alpha=1.0, zorder=5)
ax.fill_between(r_centers, mean_tot - std_tot, mean_tot + std_tot, color=total_color, alpha=0.12, zorder=4)

# ------------------------------------------------------------
# 2) AUTO terms: dotted, colored
# ------------------------------------------------------------
auto_means = {}

for name in auto_names:
    x_auto = np.asarray(xi_auto[name])
    mean_auto = np.nanmean(x_auto, axis=0)
    auto_means[name] = mean_auto

    ax.plot(r_centers, mean_auto, lw=2.2, c=auto_colors[name], ls=":", alpha=0.95, zorder=3)

# ------------------------------------------------------------
# 3) CROSS terms: dashed, colored + expected peak markers
# ------------------------------------------------------------
cross_means = []

for (name_i, name_j) in pairs:
    col = cross_colors[(name_i, name_j)]
    ls = "--"

    x_ij = np.asarray(xi_cross[name_i][name_j])
    mean_ij = np.nanmean(x_ij, axis=0)
    cross_means.append(mean_ij)

    ax.plot(r_centers, mean_ij, lw=2.2, c=col, ls=ls, alpha=0.95, zorder=2)

    # Expected cross-peak separation in the velocity/Ly-alpha frame
    lam_i = float(line_info[name_i]["lambda0"])
    lam_j = float(line_info[name_j]["lambda0"])
    expected_cross_peak = np.abs(c_kms * np.log(lam_i / lam_j))

    ax.axvline(expected_cross_peak, color=col, linestyle=ls, lw=1.6, alpha=0.55, zorder=1)

# ------------------------------------------------------------
# 4) Common y-limits
# ------------------------------------------------------------
ymins, ymaxs = [], []

ymins.append(np.nanmin(mean_tot - std_tot))
ymaxs.append(np.nanmax(mean_tot + std_tot))

for name in auto_names:
    ymins.append(np.nanmin(auto_means[name]))
    ymaxs.append(np.nanmax(auto_means[name]))

for mean_ij in cross_means:
    ymins.append(np.nanmin(mean_ij))
    ymaxs.append(np.nanmax(mean_ij))

ymin = float(np.nanmin(ymins))
ymax = float(np.nanmax(ymaxs))
pad = 0.06 * (ymax - ymin) if np.isfinite(ymax - ymin) and (ymax > ymin) else 1.0
ax.set_ylim(ymin - pad, ymax + pad)

# ------------------------------------------------------------
# 5) Cosmetics
# ------------------------------------------------------------
ax.axhline(0.0, color="k", lw=1.0, alpha=0.35)
ax.set_xlabel(r"Separation " + x_label, fontsize=16)
ax.set_ylabel(r"$\xi$", fontsize=16)

# ------------------------------------------------------------
# THREE LEGENDS
# ------------------------------------------------------------

# (1) Linestyle meaning
style_handles = [
    Line2D([0], [0], color="k", lw=2.8, ls="-",  label=r"$\xi_{\mathrm{total}}$"),
    Line2D([0], [0], color="k", lw=2.2, ls=":",  label=r"$\xi_{\mathrm{auto}}$"),
    Line2D([0], [0], color="k", lw=2.2, ls="--", label=r"$\xi_{\mathrm{cross}}$"),
]
leg_style = ax.legend(handles=style_handles, loc="upper left", frameon=True, fontsize=14, ncols=3)
ax.add_artist(leg_style)

# (2) Auto legend: colors only
auto_color_handles = [Line2D([0], [0], color=total_color, lw=5.0, ls="-", label="Total")]
for name in auto_names:
    label = line_info[name].get("label", name)
    auto_color_handles.append(
        Line2D([0], [0], color=auto_colors[name], lw=5.0, ls="-", label=label)
    )

leg_auto = ax.legend(handles=auto_color_handles, loc="lower right", frameon=True, fontsize=13, ncols=len(auto_names) + 1)
ax.add_artist(leg_auto)

# (3) Cross legend: colors only
cross_color_handles = []
for (name_i, name_j) in pairs:
    label_i = line_info[name_i].get("label", name_i)
    label_j = line_info[name_j].get("label", name_j)
    cross_color_handles.append(
        Line2D([0], [0], color=cross_colors[(name_i, name_j)], lw=5.0, ls="-", label=f"{label_i} × {label_j}")
    )

if len(cross_color_handles) > 0:
    leg_cross = ax.legend(
        handles=cross_color_handles, loc="upper right", frameon=True, fontsize=13, ncol=1,
    )
    ax.add_artist(leg_cross)